In [1]:
import pickle as pkl
import re
import toml

In [2]:
mapping_patterns = [
    (r"^I-Jockey$", "I"),
    (r"^TcMar-Pogo$", "TcMar"),
    (r"^TcMar-Tc1$", "TcMar"),
    (r"^CMC-Transib$", "CMC"),
    (r"^R1-LOA$", "R1"),
    (r"^hAT-hobo$", "hAT"),
    (r"^hAT-Tip100$", "hAT"),
    (r"^CMC-EnSpm$", "CMC"),
    (r"^RC$", "Helitron"),
    (r"^Mariner-Tc1$", "TcMar"),
    (r"^Dirs$", "DIRS"),
    (r"^Pao$", "Bel-Pao"),
    (r"^DNA$", "TIR"),
    (r"^CMC-.*$", "CACTA"),
    (r"^Tc1-.*$", "Tc1"),
    (r"^hAT-.*$", "hAT"),
    (r"^ERV.*$", "ERV"),
    (r"^L1-.*$", "L1"),
    (r"^PIF-Harbinger$", "PIF"),
    (r"^Crypton-.*$", "Crypton"),
    (r"^RTE-.*$", "RTE"),
    (r"^Retroposon$", "LINE"),
    (r"^MULE-.*$", "MULE"),
    (r"^R1-.*$", "R1"),
    (r"^tRNA-.*$", "SINE2/tRNA"),
    (r"^5S-.*$", "SINE3/5S"),
    (r"^Alu$", "SINE1/7SL"),
    (r"^B2$", "SINE2/tRNA"),
    (r"^B4$", "SINE2/tRNA"),
    (r"^MIR$", "SINE2/tRNA"),
    (r"^ID$", "SINE2/tRNA"),
    (r"^Penelope$", "PLE"),
    (r"^Unknown$", "Other"),
]

def convert_key(input_key: str) -> str:
    for pattern, target_val in mapping_patterns:
        if re.match(pattern, input_key.strip()):
            return target_val
    return input_key.strip()


In [ ]:
def read_fasta(path):
    header = None
    seq_chunks = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_chunks)
                header = line[1:]
                seq_chunks = []
            else:
                seq_chunks.append(line)

        if header is not None:
            yield header, "".join(seq_chunks)


def convert_label(order, superfamily, lhs_to_rhs, rhs_labels):
    order = (order or "").strip()
    superfamily = (superfamily or "").strip()

    # 1. exact final Terrier label
    if order and superfamily:
        candidate = f"{order}/{superfamily}"
        if candidate in rhs_labels:
            return candidate, superfamily, "exact_rhs_order_superfamily"

    # 2. map order if it exists on TOML LHS
    if order in lhs_to_rhs:
        return lhs_to_rhs[order], superfamily, "lhs_order"

    # 3. map superfamily if it exists on TOML LHS
    if superfamily in lhs_to_rhs:
        return lhs_to_rhs[superfamily], superfamily, "lhs_superfamily"

    # 4. otherwise drop
    return None, f"{order} {superfamily}", "unmapped"


def run_mapping(MAPPING_FILE, INPUT_FASTA, OUTPUT_PICKLE, UNMAPPED_PICKLE):
    lhs_to_rhs = toml.load(MAPPING_FILE)
    rhs_labels = set(lhs_to_rhs.values())

    result = {}
    unmapped = {}
    stats = {
        "total": 0,
        "mapped": 0,
        "unmapped": 0,
        "exact_rhs_order_superfamily": 0,
        "lhs_order": 0,
        "lhs_superfamily": 0,
    }

    for header, sequence in read_fasta(INPUT_FASTA):
        sequence=sequence.lower()
        sequence = re.sub(r'[^actg]', 'x', sequence) #change non actg to x
        if len(sequence) < 80: #skip if under 80 bases long
            continue

        
        stats["total"] += 1

        # Split the header to isolate the labels
        sequence_id, desc = header.split('#', 1)
        
        # Convert desc to what's in repeatmasker_labels
        desc = convert_key(desc)
        # print(desc)

        if desc == "Other":
            order, superfamily = desc, ""

        else:
            if "/" not in desc:
                desc = f"{desc}/placeholder"
            order, superfamily = desc.split("/", 1)
            order = convert_key(order)
            superfamily = convert_key(superfamily)
        # print(order, superfamily)

        mapped_label, labels, strategy = convert_label(order, superfamily, lhs_to_rhs, rhs_labels)

        if mapped_label is None:
            unmapped[sequence] = {
                "sequence_id": sequence_id,
                "order": order,
                "superfamily": superfamily,
                "header": header,
            }
            stats["unmapped"] += 1
            continue

        result[sequence] = {
            "sequence_id": sequence_id,
            "mapped_label": mapped_label,
        }
        stats["mapped"] += 1
        stats[strategy] += 1

    with open(OUTPUT_PICKLE, "wb") as f:
        pkl.dump(result, f)

    with open(UNMAPPED_PICKLE, "wb") as f:
        pkl.dump(unmapped, f)

    print("Done")
    print(f"Total: {stats['total']}")
    print(f"Mapped: {stats['mapped']}")
    print(f"Unmapped/dropped: {stats['unmapped']}")
    print("Strategy breakdown:")
    for k in ["exact_rhs_order_superfamily", "lhs_order", "lhs_superfamily"]:
        print(f"  {k}: {stats[k]}")
    print(f"Saved mapped pickle: {OUTPUT_PICKLE}")
    print(f"Saved unmapped pickle: {UNMAPPED_PICKLE}")



In [4]:
MAPPING_FILE = "repeatmasker_labels.toml"

# Drosophila

In [10]:
label = "Drosophila_melanogaster"
INPUT_FASTA = f"external_datasets/{label}.fasta"
OUTPUT_PICKLE = f"external_datasets_mapped/{label}_terrier.pkl"
UNMAPPED_PICKLE = f"external_datasets_mapped/{label}_unmapped.pkl"

run_mapping(MAPPING_FILE, INPUT_FASTA, OUTPUT_PICKLE, UNMAPPED_PICKLE)

Done
Total: 653
Mapped: 622
Unmapped/dropped: 31
Strategy breakdown:
  exact_rhs_order_superfamily: 519
  lhs_order: 85
  lhs_superfamily: 18
Saved mapped pickle: external_datasets_mapped/Drosophila_melanogaster_terrier.pkl
Saved unmapped pickle: external_datasets_mapped/Drosophila_melanogaster_unmapped.pkl


# Homo sapien

In [11]:
label = "Homo_sapiens"
INPUT_FASTA = f"external_datasets/{label}.fasta"
OUTPUT_PICKLE = f"external_datasets_mapped/{label}_terrier.pkl"
UNMAPPED_PICKLE = f"external_datasets_mapped/{label}_unmapped.pkl"
run_mapping(MAPPING_FILE, INPUT_FASTA, OUTPUT_PICKLE, UNMAPPED_PICKLE)

Done
Total: 1541
Mapped: 1355
Unmapped/dropped: 186
Strategy breakdown:
  exact_rhs_order_superfamily: 835
  lhs_order: 341
  lhs_superfamily: 179
Saved mapped pickle: external_datasets_mapped/Homo_sapiens_terrier.pkl
Saved unmapped pickle: external_datasets_mapped/Homo_sapiens_unmapped.pkl


# Mus musculus

In [12]:
label = "Mus_musculus"
INPUT_FASTA = f"external_datasets/{label}.fasta"
OUTPUT_PICKLE = f"external_datasets_mapped/{label}_terrier.pkl"
UNMAPPED_PICKLE = f"external_datasets_mapped/{label}_unmapped.pkl"
run_mapping(MAPPING_FILE, INPUT_FASTA, OUTPUT_PICKLE, UNMAPPED_PICKLE)

Done
Total: 1670
Mapped: 1530
Unmapped/dropped: 140
Strategy breakdown:
  exact_rhs_order_superfamily: 1074
  lhs_order: 282
  lhs_superfamily: 174
Saved mapped pickle: external_datasets_mapped/Mus_musculus_terrier.pkl
Saved unmapped pickle: external_datasets_mapped/Mus_musculus_unmapped.pkl


# Oryza sativa

In [13]:
label = "Oryza_sativa"
INPUT_FASTA = f"external_datasets/{label}.fasta"
OUTPUT_PICKLE = f"external_datasets_mapped/{label}_terrier.pkl"
UNMAPPED_PICKLE = f"external_datasets_mapped/{label}_unmapped.pkl"
run_mapping(MAPPING_FILE, INPUT_FASTA, OUTPUT_PICKLE, UNMAPPED_PICKLE)

Done
Total: 75
Mapped: 72
Unmapped/dropped: 3
Strategy breakdown:
  exact_rhs_order_superfamily: 65
  lhs_order: 5
  lhs_superfamily: 2
Saved mapped pickle: external_datasets_mapped/Oryza_sativa_terrier.pkl
Saved unmapped pickle: external_datasets_mapped/Oryza_sativa_unmapped.pkl


In [17]:
with open(OUTPUT_PICKLE, "rb") as f:
    data = pkl.load(f)
sequences = list(data.keys())

In [18]:
sequences

['...']

In [16]:
data

{'tgtagctccttcatacacagatccaccaaaacacaggagtagggtattacgcttctcagcggcccgaacctgtatacatcgcccgtgtctcgtgcgtttccxxtctcgcgaaccttccacaxacagcgagagcttagaatctcacccxgagcccccggccgaaccggcaaaggggggcctgcgcggtctcccggtgaggagccccgcgctccgtcatctggcgcgccaggtagggggctcxgcgtgtgtttttctgagctctcgcactctttcgtgtgcgccaagcttttcctgttcagcccaatggccgagcgagcaaaggcgcacgacctctctccgagcgcgagcggcgacgacggggagccgaacccgcgccgtcgagctcgtactccaccgccccctccacgccaaagtcctaggcggggggaggcgctcgagagggtcgaaggatccgcgacttcgccgxccgcgggcgatggaggagggcggcgagacggggagcgtcgcctcctcgtctacggcgacggcagcacgccccagggcgcgctccaggctgcgggcgcgctcctacgtcacccgcccgtcgtccctgacccggagtctccagcccaacgttggctggacgacgtggccaacttggtcatgacggcgcggcagcgcctaggtgctggtgggcggtcctccgccaccaagacctccggcgccgctaccaccggctccgcgtcgtcaaggcggagggcgcggcgagcggcggccgctgcxcgccgttcggccgccactccctcgtcgacgcctccgacccgggaagatctgcgtggggggccggatgcccgcxtcggcatcgagcgccggcgtaatggccggcgtgctgcccacgcaacggagggcgcctcctcgtccggagtgtcacctcgacacggacgcggggatcagccctccgtgcccccggttggtggtgtcggctgtagagcctttgtggcgagccttcggaacgtccg